# Testing ML Systems & CI/CD

ML test pyramids differ from software test pyramids because the behavior being tested is statistical, not deterministic. This note covers data tests, model behavioral tests, CI/CD gates, and what a GitHub Actions ML workflow looks like.

## What Interviewers Test
- ML test pyramid: data → model → integration
- Invariance tests: outputs that should not change for certain input perturbations
- Minimum functionality tests: the model must do at least X on benchmark cases
- What runs on PR vs nightly vs pre-deploy
- Model promotion gates: what metrics must pass before production

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6, random_state=42)
model = LogisticRegression(max_iter=500).fit(X, y)

# ============================================================
# LAYER 1: DATA TESTS (pytest-style)
# ============================================================
def test_schema(X, expected_n_features=10):
    """Data must have the expected number of features."""
    assert X.shape[1] == expected_n_features, f"Expected {expected_n_features} features, got {X.shape[1]}"
    return True

def test_no_nulls(X):
    """No NaN values in input features."""
    assert not np.isnan(X).any(), f"Found {np.isnan(X).sum()} NaN values"
    return True

def test_no_all_zeros(X, threshold=0.9):
    """No feature is all-zero more than threshold fraction of the time."""
    zero_rates = (X == 0).mean(axis=0)
    bad_features = np.where(zero_rates > threshold)[0]
    assert len(bad_features) == 0, f"Features {bad_features} are >90% zero"
    return True

def test_label_distribution(y, min_pos_rate=0.05, max_pos_rate=0.95):
    """Labels should not be pathologically imbalanced."""
    pos_rate = y.mean()
    assert min_pos_rate <= pos_rate <= max_pos_rate, f"pos_rate={pos_rate:.3f} outside [{min_pos_rate},{max_pos_rate}]"
    return True

print("=== Data Tests ===")
for test_fn in [test_schema, test_no_nulls, test_no_all_zeros]:
    result = test_fn(X)
    print(f"  {test_fn.__name__}: {'PASS' if result else 'FAIL'}")
print(f"  test_label_distribution: {'PASS' if test_label_distribution(y) else 'FAIL'}")


In [ ]:
# ============================================================
# LAYER 2: MODEL BEHAVIORAL TESTS
# ============================================================
from sklearn.metrics import roc_auc_score

def test_minimum_auc(model, X, y, min_auc=0.7):
    """Model must achieve at least min_auc on evaluation data."""
    auc = roc_auc_score(y, model.predict_proba(X)[:,1])
    assert auc >= min_auc, f"AUC={auc:.4f} < minimum {min_auc}"
    return auc

def test_invariance_irrelevant_feature(model, X, feature_idx, n_samples=100):
    """Perturbing an irrelevant feature should not significantly change predictions."""
    X_test = X[:n_samples].copy()
    preds_original = model.predict_proba(X_test)[:,1]
    
    X_perturbed = X_test.copy()
    X_perturbed[:, feature_idx] = np.random.randn(n_samples) * 1000  # extreme perturbation
    preds_perturbed = model.predict_proba(X_perturbed)[:,1]
    
    # If feature is truly irrelevant, predictions should be similar
    mean_diff = np.abs(preds_original - preds_perturbed).mean()
    return mean_diff

def test_directional_expectation(model, X, feature_idx, expected_direction='positive'):
    """Increasing a feature should move predictions in expected direction."""
    X_base = X[:100].copy()
    X_high = X_base.copy()
    X_high[:, feature_idx] += 2.0  # increase feature by 2 std
    
    pred_base = model.predict_proba(X_base)[:,1].mean()
    pred_high = model.predict_proba(X_high)[:,1].mean()
    
    if expected_direction == 'positive':
        assert pred_high > pred_base, f"Feature {feature_idx} expected positive effect"
    else:
        assert pred_high < pred_base, f"Feature {feature_idx} expected negative effect"
    return pred_high - pred_base

def test_minimum_functionality(model):
    """Model must classify obvious examples correctly."""
    # Create unambiguously positive example (all features very high)
    X_easy_pos = np.ones((5, X.shape[1])) * 3
    X_easy_neg = np.ones((5, X.shape[1])) * -3
    preds_pos = model.predict_proba(X_easy_pos)[:,1]
    preds_neg = model.predict_proba(X_easy_neg)[:,1]
    assert preds_pos.mean() > 0.5, "Should classify obvious positives correctly"
    assert preds_neg.mean() < 0.5, "Should classify obvious negatives correctly"
    return True

print("=== Model Behavioral Tests ===")
auc = test_minimum_auc(model, X, y, min_auc=0.70)
print(f"  minimum_auc: PASS (AUC={auc:.4f})")

try:
    delta = test_directional_expectation(model, X, feature_idx=0, expected_direction='positive')
    print(f"  directional_expectation feat_0: PASS (delta={delta:+.4f})")
except AssertionError as e:
    print(f"  directional_expectation feat_0: FAIL ({e})")

print(f"  minimum_functionality: {'PASS' if test_minimum_functionality(model) else 'FAIL'}")


## CI/CD Pipeline for ML

```yaml
# .github/workflows/ml_ci.yml (in markdown — for reference)

# PR checks (fast, < 5 min):
#   - Data schema validation
#   - Unit tests for feature engineering
#   - Model behavioral tests on small eval set
#   - Lint and type checks

# Nightly (< 30 min):
#   - Full eval on holdout set
#   - Regression tests vs production model baseline
#   - Data drift check on recent traffic sample
#   - Training smoke test (1 epoch, small data)

# Pre-deploy (manual trigger, < 2 hr):
#   - Full training pipeline run
#   - Shadow evaluation on production traffic
#   - Performance benchmarks (latency, memory)
#   - Model promotion gate check
```


## Model Promotion Gate

Before promoting from staging to production, ALL must pass:
- AUC ≥ production model baseline (no regression)
- Latency p99 < SLA (e.g., 50ms)
- Memory footprint ≤ serving budget
- No test failures in behavioral suite
- Data schema compatibility verified
- Shadow test: score distribution PSI < 0.1 vs production


## Common Interview Questions

**Q: What is an invariance test for an ML model?**
An invariance test verifies that model outputs are stable when inputs that should not matter are changed. For a text classifier, perturbing whitespace should not change predictions. For a fraud model, changing an uninformative feature ID should not change scores significantly. These catch bugs where the model learned spurious correlations from training data.

**Q: What is a directional expectation test?**
A test that verifies the model's outputs move in the expected direction when a feature changes. For a credit risk model: increasing income should decrease predicted default probability. These are unit tests for model behavior that can catch training bugs, feature engineering errors, and sign flips in preprocessing.

**Q: What should run on a PR vs nightly vs pre-deploy?**
PR: fast tests (< 5 min) — schema tests, unit tests, small eval set behavioral tests. Nightly: slower tests — full eval, regression vs baseline, drift check. Pre-deploy: full training pipeline, shadow eval, promotion gate. The rule: gate on PR tests (block merge if fail) but not nightly/pre-deploy (fix asynchronously).

**Q: What is a model promotion gate?**
A set of automated checks that must all pass before a model can be promoted from staging to production. Typically: no AUC regression vs current production, latency and memory within bounds, no behavioral test failures, schema compatibility verified. This prevents accidental regressions from reaching users.

## Key Takeaways
- ML test pyramid: data tests (schema, distributions) → model tests (behavioral, invariance) → integration tests
- Invariance tests: outputs should not change for semantically irrelevant perturbations
- Directional tests: model must respond in the expected direction to key features
- Minimum functionality tests: the model must correctly handle a set of clear-cut cases
- CI gates: PR = fast unit/behavioral tests; nightly = full eval + drift; pre-deploy = full pipeline + shadow
- Promotion gate: AUC ≥ baseline, latency ≤ SLA, memory ≤ budget, schema compatible — all must pass